# SmartDs Radial Histograms (Old-Style, Directly Computable)

This notebook reproduces some of the old batplotlib-style histogram/radius plots using the **existing library helpers** in `starwinds_analysis.visualisation.histograms`.

- It uses a `SmartDs` object for data access.
- It does **not** resample or build shells.
- These are direct point-based diagnostics from the native 3D BATSRUS output.

## Load A Sample 3D File

Default: `sample_data/3d__var_3_n00060000.plt`.

The BATSRUS SI graph is attached so we can also plot SI-converted fields where available.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from starwinds_analysis.data.samples import get_sample
from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.visualisation.histograms import (
    plot_binned_vs_radius,
    plot_cumulative_hists,
    plot_radial_hist2d,
    plot_vs_radius,
)

plt.rcParams["figure.dpi"] = 120
STAR_RADIUS_M = 6.957e8

In [ ]:
sample_path = get_sample("3d__var_1_n00060000.plt")
sds = SmartDs.from_file(sample_path)
sds.add_batsrus_graph(body_radius_m=STAR_RADIUS_M)

sample_path

In [ ]:
def keep_available(ds, fields):
    return tuple(f for f in fields if ds.has_field(f))

raw_fields = keep_available(
    sds,
    (
        "Rho [g/cm^3]",
        "P [dyne/cm^2]",
        "E [erg/cm^3]",
        "ti [K]",
    ),
)

si_fields = keep_available(
    sds,
    (
        "Rho [kg/m^3]",
        "P [Pa]",
        "E [J/m^3]",
        "M_A [none]",
    ),
)

print("Raw fields:", raw_fields)
print("SI/derived fields:", si_fields)

## Cumulative Histograms (CDF)

These use the old-style `plot_cumulative_hists(...)` helper directly on the `SmartDs` object.

In [ ]:
# DONE symlog scale for y axis similar to the quicklook profiles.
fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
plot_cumulative_hists(sds, axs, fields=raw_fields, bins=160)
for ax in np.array(axs).ravel():
    ax.set_yscale("symlog", linthresh=1e-2)
    ax.grid(True, alpha=0.3)
    ax.grid(True, which="minor", alpha=0.1)
fig.suptitle("CDFs from native BATSRUS point data", y=1.02)
plt.show()


## Radius Scatter (Point Cloud)

This is the direct point-based `field vs r` view (no shell averaging).

In [ ]:
# DONE use symlog scale for x asis as above.
fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
plot_vs_radius(sds, axs, fields=raw_fields, s=0.15, alpha=0.15)
for ax in np.array(axs).ravel():
    ax.set_xscale("symlog", linthresh=1e-2)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.grid(True, which="minor", alpha=0.1)
fig.suptitle("Raw fields vs radius (point cloud)", y=1.02)
plt.show()


## Binned Radial Profiles (SI / Derived Where Available)

This uses `plot_binned_vs_radius(...)` on SI-converted and derived fields provided through `SmartDs + griblet`.

In [ ]:
#DONE log axis on y, symolg on x as above.
fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
plot_binned_vs_radius(sds, axs, fields=si_fields, bins=96, statistic="mean")
for ax in np.array(axs).ravel():
    ax.set_xscale("symlog", linthresh=1e-2)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.grid(True, which="minor", alpha=0.1)
fig.suptitle("Binned radial means (SI-first fields)", y=1.02)
plt.show()


## 2D Radius Histograms ("Monster" Replacement)

Compact replacement for the old monster histogram: radius on x, field value on y, color shows counts (or per-radius normalized counts).

In [ ]:
#DONE log axis on y, symolg on x as above.

fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
plot_radial_hist2d(
    sds,
    axs,
    fields=raw_fields,
    bins=(96, 96),
    normalize="per_radius",
    log_color=True,
)
for ax in np.array(axs).ravel():
    ax.set_xscale("symlog", linthresh=1e-2)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.grid(True, which="minor", alpha=0.1)
fig.suptitle("Radius-value hist2d (old monster-style, compact)", y=1.02)
plt.show()


## Notes

- These helpers operate on any dataset-like object callable as `ds(field_name)`, so `SmartDs` works directly.
- Raw-unit plotting is fine here for quick inspection.
- For analytical quantities (mass loss, torque, shell fluxes), use the SI-first analysis functions instead.